# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yumna-09/FlyRank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Before writing any rule, I check the signals it would lean on. Two checks below — both are tied to real FlyRank product flags (`DATA_USE.md` / `ml-intern-dataset-and-lane-guide.md`): **staleness** sits behind the refresh flags, and **CTR-vs-position** sits behind the CTR-fix logic. Each check gets a bucket table with `n` printed, and a one-word verdict.

### Signal check 1 (flag-linked — refresh flags): does staleness predict decline?

FlyRank's refresh flags assume a stale page (not updated in a long time) is more likely to be declining. I test that directly: bucket pages by `freshness_tier` and look at the `is_declining_label` rate in each bucket, across the whole dataset (not just my lane's slice) so the buckets have real `n`.

In [1]:
import os
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")
print("Working dir:", os.getcwd())

import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

signal1 = (
    df.groupby("freshness_tier")
      .agg(n=("content_id", "size"), decline_rate=("is_declining_label", "mean"))
      .reindex(["0-30", "31-90", "91-180", "181+"])
)
signal1["decline_rate"] = signal1["decline_rate"].round(3)
print(f"overall decline rate (all {len(df):,} pages): {df['is_declining_label'].mean():.3f}\n")
print(signal1)

Working dir: /home/claude/repo


overall decline rate (all 30,000 pages): 0.542

                    n  decline_rate
freshness_tier                     
0-30            20480         0.511
31-90             175         0.589
91-180           9171         0.611
181+              174         0.471


**Verdict: MIXED.**

Decline rate rises from 0-30 days (51.1%, n=20,480) to 91-180 days (61.1%, n=9,171) — that part points the way the refresh flag assumes. But the 181+ bucket (n=174) drops back to 47.1%, *below* the 51.4% dataset average — the opposite of what "more stale = more likely declining" predicts. That bucket is small, but it's not noise I can wave away: it directly contradicts the pattern the bigger buckets suggested. Staleness alone is not a safe decline predictor here — a clearly-explained negative, and it saves the rule: **I leave staleness out of the score below** rather than build on a signal that reverses itself.

### Signal check 2 (flag-linked — CTR-fix logic): does CTR fall as position worsens?

FlyRank's "needs CTR fix" logic assumes each position band has a typical CTR, and a page getting far less than that typical CTR is worth a CTR review. I test the first half of that assumption: does CTR reliably drop as `position_tier` gets worse? I restrict to visible pages (`impressions_90d >= 500`) so low-n noise doesn't drive the averages — the data dictionary itself warns that `top_3` pages get very few impressions and small samples swing CTR hard.

In [2]:
visible = df[(df["impressions_90d"] >= 500) & (df["position_tier"] != "no_data") & (df["avg_position"] > 0)]
order = ["top_3", "page_1", "striking", "page_3_5", "deep"]

signal2 = (
    visible.groupby("position_tier")
           .agg(n=("content_id", "size"), mean_ctr=("ctr", "mean"), median_ctr=("ctr", "median"))
           .reindex(order)
)
print(f"n visible pages with real position data: {len(visible):,}\n")
print(signal2.round(3))

n visible pages with real position data: 16,726

                  n  mean_ctr  median_ctr
position_tier                            
top_3           458     0.347        0.20
page_1         7064     0.339        0.24
striking       4485     0.267        0.17
page_3_5       4330     0.143        0.09
deep            389     0.043        0.00


**Verdict: CONFIRMED.**

Median CTR falls in lock-step with position, every bucket well-sampled (n=389 to n=7,064): `top_3` 0.20% → `page_1` 0.24% → `striking` 0.17% → `page_3_5` 0.09% → `deep` 0.00%. (`top_3`'s mean sits a touch below `page_1`'s despite ranking higher — exactly the small-sample CTR noise the data dictionary warns about at n=458 — so I use each bucket's **median** as the "typical CTR" benchmark, not the mean.) The pattern is real: position genuinely drives expected CTR, so a page sitting well below its own bucket's typical CTR is a legitimate signal, not a coincidence.

### The rule, in plain words

**A page is worth a CTR review when it is visible enough to matter, ranks well enough that clicks should be flowing, but its CTR badly lags what pages at that same position typically earn. The bigger the gap, and the more visible the page, the higher the priority.**

Staleness was tested and dropped (Signal check 1, MIXED) — it doesn't reliably predict decline here, so it doesn't belong in the score. CTR-vs-position (Signal check 2, CONFIRMED) is the real driver, paired with visibility (volume) so a huge CTR gap on a page nobody sees still ranks low.

**Reason code (one, applied everywhere the same way):** `ctr_below_position_expected` — the page's actual CTR sits below the median CTR for its position tier.

**Score (transparent, no fitted weights):**
```
eligible   = impressions_90d >= 500  AND  avg_position > 0  AND  position_tier != 'no_data'
ctr_gap    = max(median_ctr_for_position_tier - ctr, 0)      # 0 when page meets/beats its tier
visibility = percentile_rank(log1p(impressions_90d))         # 0-1, volume signal
gap_norm   = percentile_rank(ctr_gap)                        # 0-1
score      = eligible * visibility * gap_norm
```
**Action label (threshold on score, not fitted):** `score >= 0.6` → `prioritize_ctr_review`; `0 < score < 0.6` → `review_ctr`; `score == 0` (includes ineligible pages) → `monitor`.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
import os

# Whole 30k-row starter slice — no future-window columns, no product flags.
expected_ctr_by_tier = visible.groupby("position_tier")["ctr"].median()

eligible = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["position_tier"] != "no_data")
)

df["expected_ctr"] = df["position_tier"].map(expected_ctr_by_tier)
df["ctr_gap"] = (df["expected_ctr"] - df["ctr"]).clip(lower=0)

def percentile_rank(s):
    return s.rank(method="average", pct=True)

df["ctr_gap_norm"] = percentile_rank(df["ctr_gap"].where(eligible, 0.0))
df["visibility_score"] = percentile_rank(np.log1p(df["impressions_90d"]))
df["baseline_action_score"] = np.where(eligible, df["visibility_score"] * df["ctr_gap_norm"], 0.0).round(4)

df["reason_code"] = "ctr_below_position_expected"

def action_label(score):
    if score >= 0.6:
        return "prioritize_ctr_review"
    if score > 0:
        return "review_ctr"
    return "monitor"

df["action"] = df["baseline_action_score"].apply(action_label)

print(df["action"].value_counts())
print()
print("score describe:")
print(df["baseline_action_score"].describe().round(3))

queue = df.sort_values("baseline_action_score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

out_cols = [
    "rank", "content_id", "client_id",
    "baseline_action_score", "reason_code", "action",
    "impressions_90d", "avg_position", "position_tier", "ctr", "expected_ctr", "ctr_gap",
    "visibility_score", "trend_direction",
]

os.makedirs("work/outputs", exist_ok=True)
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"\nwrote work/outputs/baseline_action_score.csv — {len(queue):,} rows")


action
monitor                  13274
review_ctr               12814
prioritize_ctr_review     3912
Name: count, dtype: int64

score describe:
count    30000.000
mean         0.241
std          0.261
min          0.000
25%          0.000
50%          0.208
75%          0.413
max          0.991
Name: baseline_action_score, dtype: float64



wrote work/outputs/baseline_action_score.csv — 30,000 rows


## 3. Top-10 review

*For each of my top ten: the action, why it's there, and what would make it wrong.* (The card asks for top-10; the sibling `w04_signal_audit.ipynb` skeleton has room for a top-20 if I want to go further later — not required here.)

In [5]:
top10 = queue.head(10)
top10[["rank", "content_id", "client_id", "baseline_action_score", "action",
       "impressions_90d", "position_tier", "avg_position", "ctr", "expected_ctr", "ctr_gap",
       "trend_direction"]]

,rank,content_id,client_id,baseline_action_score,action,impressions_90d,position_tier,avg_position,ctr,expected_ctr,ctr_gap,trend_direction
0,1,content_c8e9d6ab9013,client_19581e27de,0.9909,prioritize_ctr_review,208678,page_1,9.7,0.00,0.24,0.24,down
1,2,content_453722754fea,client_f369cb89fc,0.9808,prioritize_ctr_review,140079,page_1,7.6,0.01,0.24,0.23,down
2,3,content_c84a0ab98e90,client_f369cb89fc,0.9796,prioritize_ctr_review,223271,page_1,7.8,0.03,0.24,0.21,stable
3,4,content_39881853ef0c,client_f369cb89fc,0.9788,prioritize_ctr_review,112434,page_1,7.2,0.01,0.24,0.23,down
4,5,content_0919dd345d80,client_4e07408562,0.9786,prioritize_ctr_review,119217,page_1,7.0,0.02,0.24,0.22,down
5,6,content_c1fe78bc4e37,client_19581e27de,0.9777,prioritize_ctr_review,134055,page_1,7.5,0.03,0.24,0.21,down
6,7,content_b115f7c74779,client_19581e27de,0.9766,prioritize_ctr_review,123469,page_1,8.0,0.03,0.24,0.21,up
7,8,content_63f88d16fdb8,client_19581e27de,0.9747,prioritize_ctr_review,99013,page_1,6.4,0.03,0.24,0.21,down
8,9,content_d0cc5baa4995,client_19581e27de,0.9728,prioritize_ctr_review,83651,page_1,6.6,0.03,0.24,0.21,down
9,10,content_36ff89c8214e,client_19581e27de,0.9723,prioritize_ctr_review,295097,page_1,7.3,0.05,0.24,0.19,stable


**Top-10, one line each:**

1. **content_c8e9d6ab9013** — `prioritize_ctr_review`. page_1 (avg pos 9.7), 208,678 impressions, CTR 0.00% vs 0.24% expected — a near-total CTR gap on a huge, well-ranked page. *Wrong if:* the page has a noindex/redirect issue GSC hasn't caught, so "no clicks" is a tracking artifact, not a content problem.
2. **content_453722754fea** — `prioritize_ctr_review`. page_1 (7.6), 140,079 impressions, CTR 0.01% vs 0.24%. *Wrong if:* the title/meta already changed recently and GSC hasn't caught up — the fix may already be shipped.
3. **content_c84a0ab98e90** — `prioritize_ctr_review`. page_1 (7.8), 223,271 impressions, CTR 0.03% vs 0.24%, `trend_direction = stable` not declining. *Wrong if:* stable-but-low-CTR is this page's normal state (e.g. branded query where users already know the answer from the snippet) — reviewing it may find nothing to fix.
4. **content_39881853ef0c** — `prioritize_ctr_review`. page_1 (7.2), 112,434 impressions, CTR 0.01% vs 0.24%. *Wrong if:* the SERP snippet is being outranked visually by a featured snippet or ad block that a title/meta rewrite can't fix.
5. **content_0919dd345d80** — `prioritize_ctr_review`. page_1 (7.0), 119,217 impressions, CTR 0.02% vs 0.24%. *Wrong if:* this is a very recent ranking jump and the 90-day CTR average hasn't caught up to the new (better) position yet.
6. **content_c1fe78bc4e37** — `prioritize_ctr_review`. page_1 (7.5), 134,055 impressions, CTR 0.03% vs 0.24%. *Wrong if:* the query mix behind these impressions is mostly navigational (users searching the brand name directly) — low CTR there doesn't mean a bad title/meta.
7. **content_b115f7c74779** — `prioritize_ctr_review`. page_1 (8.0), 123,469 impressions, CTR 0.03% vs 0.24%, `trend_direction = up`. *Wrong if:* impressions are climbing faster than clicks can keep pace with — a CTR gap during rapid growth is a different (probably self-resolving) problem than a genuinely broken snippet.
8. **content_63f88d16fdb8** — `prioritize_ctr_review`. page_1 (6.4), 99,013 impressions, CTR 0.03% vs 0.24%. *Wrong if:* `search_volume`/`main_intent` show this is a low-intent informational query where low CTR is expected regardless of snippet quality.
9. **content_d0cc5baa4995** — `prioritize_ctr_review`. page_1 (6.6), 83,651 impressions, CTR 0.03% vs 0.24%. *Wrong if:* `content_type` is one with heavy missing keyword data (the dictionary flags this) — no keyword context makes it hard to know what to even rewrite the title toward.
10. **content_36ff89c8214e** — `prioritize_ctr_review`. page_1 (7.3), 295,097 impressions, CTR 0.05% vs 0.24%, `trend_direction = stable`. *Wrong if:* this client's whole portfolio runs low CTR (a client-level pattern, not a page-level one) — reviewing one page in isolation would miss the real fix.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
# Which top-10 picks look weakest, and why (data-driven, not just my hand-wave above)?
weak_flags = top10.copy()
weak_flags["not_declining"] = weak_flags["trend_direction"].isin(["stable", "up"])
print("Top-10 rows where trend_direction is NOT 'down':")
print(weak_flags.loc[weak_flags["not_declining"],
      ["rank", "content_id", "trend_direction", "ctr", "expected_ctr"]])

print()
# Leakage check: confirm the score/queue never touches trend_direction or trend_pct
# (the label source) or any FlyRank product decision flag as an INPUT.
score_inputs = ["impressions_90d", "avg_position", "position_tier", "ctr"]
label_source = ["trend_direction", "trend_pct", "is_declining_label"]
product_flags = ["health_score", "priority_score", "action_type", "needs_ctr_fix", "is_quick_win"]

print("Score built from:", score_inputs)
print("Label-source columns (never used as inputs):", label_source)
print("Product flags in this dataset at all?",
      [c for c in product_flags if c in df.columns] or "none present — starter CSV ships observable signals only")
print("trend_direction used anywhere in the score/action logic?",
      any(col in ("trend_direction", "trend_pct") for col in score_inputs))

Top-10 rows where trend_direction is NOT 'down':
   rank            content_id trend_direction   ctr  expected_ctr
2     3  content_c84a0ab98e90          stable  0.03          0.24
6     7  content_b115f7c74779              up  0.03          0.24
9    10  content_36ff89c8214e          stable  0.05          0.24

Score built from: ['impressions_90d', 'avg_position', 'position_tier', 'ctr']
Label-source columns (never used as inputs): ['trend_direction', 'trend_pct', 'is_declining_label']
Product flags in this dataset at all? none present — starter CSV ships observable signals only
trend_direction used anywhere in the score/action logic? False


**Weak picks:** 3 of the top 10 don't carry `trend_direction = down` — #3 and #10 are `stable`, #7 is `up`. Three rows out of ten where the page isn't even labeled declining, despite scoring highest on the CTR-gap rule. That's expected and healthy: this rule was never built to predict decline (I dropped staleness *because* it didn't reliably predict decline), it targets a different, narrower thing — "CTR badly lags what this position normally earns." A stable or growing page can still have a real CTR problem; a hand review should confirm the SERP snippet itself before assuming the rule is wrong about these three.

**No future-window or label-derived inputs:** the score uses only `impressions_90d`, `avg_position`, `position_tier`, and `ctr` — all trailing-90-day observed metrics, none of them derived from `trend_direction` or `trend_pct` (the label source per the data dictionary), and none of them a FlyRank product decision flag (those aren't even shipped in this starter CSV — confirmed above). `content_id` / `client_id` are used for identification only, never as score inputs.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.